In [ ]:
import numpy as np
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score
from itertools import product
from typing import Dict, List, Tuple
import pandas as pd
import os
import warnings
from dataclasses import dataclass
from tqdm import tqdm

warnings.filterwarnings('ignore')

# =============================================================================
# 1. CONFIGURATION & GLOBAL CONTROLS
# =============================================================================
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"
OUTPUT_FILE = "./global_optimized_isotope_results.csv"

MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common_synced.h5ad", "halfbrain_yc_2_filtered_common_synced.h5ad",
    "halfbrain_yc_3_filtered_common_synced.h5ad", "halfbrain_yc_4_filtered_common_synced.h5ad",
    "halfbrain_yad_1_filtered_common_synced.h5ad", "halfbrain_yad_2_filtered_common_synced.h5ad",
    "halfbrain_yad_3_filtered_common_synced.h5ad", "halfbrain_yad_4_filtered_common_synced.h5ad",
    "halfbrain_ac_1_filtered_common_synced.h5ad", "halfbrain_ac_2_filtered_common_synced.h5ad",
    "halfbrain_ac_3_filtered_common_synced.h5ad", "halfbrain_ac_4_filtered_common_synced.h5ad",
    "halfbrain_aad_1_filtered_common_synced.h5ad", "halfbrain_aad_2_filtered_common_synced.h5ad",
    "halfbrain_aad_3_filtered_common_synced.h5ad", "halfbrain_aad_4_filtered_common_synced.h5ad"
]

# Control pairs for global calibration
POSITIVE_CONTROLS = [("760.58", "761.58"), ("788.62", "789.62"), ("810.60", "811.60"), ("792.55", "793.55")]
NEGATIVE_CONTROLS = [("760.58", "810.60"), ("788.62", "114.04"), ("810.60", "792.55")]

MASS_DIFFS = {'M+1': 1.0033, 'M+2': 2.0067, 'Na': 21.982, 'K': 37.9555}
TOL = 0.01

@dataclass
class SpatialSignature:
    mz: str; values: np.ndarray; histogram: np.ndarray; importance: np.ndarray; morans: float

# =============================================================================
# 2. CALCULATION ENGINE (8 METRICS)
# =============================================================================

def get_8_metrics(s1: SpatialSignature, s2: SpatialSignature) -> Dict[str, float]:
    # 1. Intensity Ratio Consistency
    mask = (s1.values > np.percentile(s1.values, 10)) & (s2.values > np.percentile(s2.values, 10))
    r_con = 0
    if mask.sum() > 10:
        rats = np.minimum(s1.values[mask], s2.values[mask]) / (np.maximum(s1.values[mask], s2.values[mask]) + 1e-8)
        r_con = 1 / (1 + (np.std(rats) / (np.mean(rats) + 1e-8)))
    
    # Metrics based on Pearson Correlation
    v_corr = max(0, pearsonr(s1.values, s2.values)[0])
    h_corr = max(0, pearsonr(s1.histogram.flatten(), s2.histogram.flatten())[0])
    i_corr = max(0, pearsonr(s1.importance, s2.importance)[0])
    
    # Peak and IoU metrics
    p1, p2 = s1.values >= np.percentile(s1.values, 80), s2.values >= np.percentile(s2.values, 80)
    peak = (p1 & p2).sum() / ((p1 | p2).sum() + 1e-8)
    m_sim = 1 / (1 + abs(s1.morans - s2.morans))
    imp_iou = ((s1.importance > 0.5) & (s2.importance > 0.5)).sum() / ((s1.importance > 0.5) | (s2.importance > 0.5)).sum()
    val_iou = np.minimum(s1.values, s2.values).sum() / (np.maximum(s1.values, s2.values).sum() + 1e-8)
    
    return {'ratio': r_con, 'v_corr': v_corr, 'h_corr': h_corr, 'i_corr': i_corr,
            'peak': peak, 'm_sim': m_sim, 'imp_iou': imp_iou, 'val_iou': val_iou}

# =============================================================================
# 3. GLOBAL PIPELINE
# =============================================================================

class GlobalThesisPipeline:
    def __init__(self):
        self.global_weights = {}

    def get_sig(self, adata, mz, nn_idx):
        v = adata[:, mz].X.toarray().flatten() if hasattr(adata.X, 'toarray') else adata[:, mz].X.flatten()
        coords = np.column_stack([adata.obs['x_um'], adata.obs['y_um']])
        hist = np.histogram2d(coords[:,0], coords[:,1], bins=10, weights=v)[0]
        imp = np.var(v[nn_idx[:, 1:]], axis=1)
        return SpatialSignature(mz, v, hist, imp, 0.5)

    def calibrate_globally(self):
        print("--- PHASE 1: POOLING DATA FROM ALL 16 SAMPLES FOR CALIBRATION ---")
        aggregated_y_true = []
        aggregated_metrics = []

        for file_name in tqdm(MSI_SAMPLE_FILES, desc="Extracting Controls"):
            path = os.path.join(MSI_INPUT_FOLDER, file_name)
            adata = sc.read_h5ad(path)
            coords = np.column_stack([adata.obs['x_um'], adata.obs['y_um']])
            nn_idx = NearestNeighbors(n_neighbors=9).fit(coords).kneighbors(coords)[1]

            for m1, m2 in POSITIVE_CONTROLS + NEGATIVE_CONTROLS:
                if m1 in adata.var_names and m2 in adata.var_names:
                    s1, s2 = self.get_sig(adata, m1, nn_idx), self.get_sig(adata, m2, nn_idx)
                    aggregated_metrics.append(get_8_metrics(s1, s2))
                    aggregated_y_true.append(1 if (m1, m2) in POSITIVE_CONTROLS else 0)

        print(f"Total calibration points: {len(aggregated_metrics)}")
        
        print("--- PHASE 2: GLOBAL WEIGHT OPTIMIZATION (GRID SEARCH) ---")
        best_auc, best_w = -1, {}
        # Searching through primary indicator weights
        for r_w, v_w, h_w in product([0.25, 0.35], [0.15, 0.20], [0.15, 0.20]):
            rem = 1.0 - (r_w + v_w + h_w)
            if rem < 0.05: continue
            oth = rem / 5
            
            y_pred = [r_w*m['ratio'] + v_w*m['v_corr'] + h_w*m['h_corr'] + 
                      oth*(m['i_corr'] + m['peak'] + m['m_sim'] + m['imp_iou'] + m['val_iou']) for m in aggregated_metrics]
            
            auc = roc_auc_score(aggregated_y_true, y_pred)
            if auc > best_auc:
                best_auc = auc
                best_w = {'ratio': r_w, 'v_corr': v_w, 'h_corr': h_w, 'other': oth}
        
        self.global_weights = best_w
        print(f"Global Optimization Finished. AUC: {best_auc:.4f}\nOptimal Weight Set: {best_w}\n")

    def run_final_analysis(self):
        self.calibrate_globally()
        print("--- PHASE 3: FINAL ANALYSIS ACROSS ALL SAMPLES ---")
        all_samples_data = []

        for file_name in tqdm(MSI_SAMPLE_FILES, desc="Analyzing All Samples"):
            adata = sc.read_h5ad(os.path.join(MSI_INPUT_FOLDER, file_name))
            sid = file_name.split('_filtered')[0] # Clean sample ID
            coords = np.column_stack([adata.obs['x_um'], adata.obs['y_um']])
            nn_idx = NearestNeighbors(n_neighbors=9).fit(coords).kneighbors(coords)[1]
            mzs = adata.var_names.astype(float)

            for i in range(len(mzs)):
                for j in range(i + 1, len(mzs)):
                    diff = abs(mzs[j] - mzs[i])
                    if any(abs(diff - t) < TOL for t in MASS_DIFFS.values()):
                        s1, s2 = self.get_sig(adata, adata.var_names[i], nn_idx), self.get_sig(adata, adata.var_names[j], nn_idx)
                        m = get_8_metrics(s1, s2)
                        w = self.global_weights
                        score = (w['ratio']*m['ratio'] + w['v_corr']*m['v_corr'] + w['h_corr']*m['h_corr'] + 
                                 w['other']*(m['i_corr'] + m['peak'] + m['m_sim'] + m['imp_iou'] + m['val_iou']))
                        
                        all_samples_data.append({
                            'sample_id': sid, 'mz_1': adata.var_names[i], 'mz_2': adata.var_names[j],
                            'mass_diff': round(diff, 4), 'similarity_score': round(score, 4)
                        })

        pd.DataFrame(all_samples_data).to_csv(OUTPUT_FILE, index=False)
        print(f"\nSUCCESS: Master results saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    GlobalThesisPipeline().run_final_analysis()